In [3]:
%pip install --upgrade --quiet "google-cloud-aiplatform==1.152.0"

Note: you may need to restart the kernel to use updated packages.


In [10]:
PROJECT_ID = "gcp-sandbox-kwlee"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "us-central1"

In [11]:
# 스킬 목록 확인 (https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/skill-registry/create-manage#list-skills)
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

pager = client.skills.list()
for skill in pager:
    print(skill.name, skill.display_name)

projects/458778613248/locations/us-central1/skills/gcp-skill-registry gcp-skill-registry


In [16]:
# GitHub 에 있는 SKILL 등록을 위해 주소 (
GITHUB_REPO_URL = "https://github.com/kiwonilee/agentic-design-patterns-skills" # @param {type:"string"}

In [17]:
class Skill:
  def __init__(self, name, description, local_skill_dir):
    self.name = name
    self.description = description
    self.local_skill_dir = local_skill_dir

  def __str__(self):
    return f"Skill(name={self.name}, description={self.description}, local_skill_dir={self.local_skill_dir})"

def parse_skill_md(filepath: str) -> tuple[str, str]:
  """Parses SKILL.md file to get the skill name and description."""
  name = "Untitled Skill"
  description = "No description provided."
  try:
    with open(filepath, "r", encoding="utf-8") as f:
      content = f.read()
      if content.startswith("---"):
        end_idx = content.find("---", 3)
        if end_idx != -1:
          yaml_part = content[3:end_idx]
          try:
            data = yaml.safe_load(yaml_part)
            if data:
              name = data.get("name", name)
              description = data.get("description", description)
          except yaml.YAMLError as yaml_e:
            print(f"YAML parsing warning (using fallback): {yaml_e}")
            match_name = re.search(r"^name:\s*(.+)$", yaml_part, re.M)
            if match_name:
              name = match_name.group(1).strip()
            match_desc = re.search(
                r"^description:\s*(?:>-\s*)?(.+)$", yaml_part, re.M
            )
            if match_desc:
              description = match_desc.group(1).strip()
  except IOError as e:
    print(f"Failed to parse {filepath}: {e}")
  return name, description

def get_all_skills_from_dir(repo_dir: str) -> list[Skill]:
  """Returns a list of Skill objects from the given directory."""
  skills = []
  skills_found = 0
  seen_skills = set()
  for root, dirs, files in os.walk(repo_dir):
    lower_files = {f.lower(): f for f in files}
    if "skill.md" in lower_files:
      # Prune subdirectories to avoid deeper recursion in this skill folder.
      dirs[:] = []

      skill_md_filename = lower_files["skill.md"]
      filepath = os.path.join(root, skill_md_filename)

      name, description = parse_skill_md(filepath)

      # De-dupe based on skill name
      if name in seen_skills:
        print(f"Skipping duplicate skill name: {name}")
        continue
      seen_skills.add(name)

      skills.append(Skill(name, description, root))
      skills_found += 1
  print(f"Found {skills_found} skills.")
  return skills

def download_github_repo(repo_url, output_dir):
  os.makedirs(output_dir, exist_ok=True)
  branch = "master"

  zip_url = f"{GITHUB_REPO_URL}/archive/refs/heads/{branch}.zip"
  print(f"Attempting to download repository zip via HTTP: {zip_url}...")

  with tempfile.NamedTemporaryFile(suffix=".zip", delete=False) as temp_zip_file:
    zip_path = temp_zip_file.name

  request = urllib.request.Request(zip_url)
  with urllib.request.urlopen(request, timeout=30) as response:
    with open(zip_path, "wb") as out_file:
      shutil.copyfileobj(response, out_file)
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)
  print(
      f"Repository extracted successfully from '{branch}' branch zip"
      " archive."
  )
  os.remove(zip_path)
  print(f"Deleted temporary zip file: {zip_path}")

def create_skill(skill):
  name = skill.name
  description = skill.description
  skill_dir = skill.local_skill_dir
  print(f"Creating skill: {skill}")
  timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
  skill = client.skills.create(
      display_name=name,
      description=description,
      config={
          "local_path": skill_dir,
          "skill_id": f"{name}-{timestamp}"
      }
  )
  print(f"Created skill: {name}")

In [19]:
output_dir = tempfile.mkdtemp()
download_github_repo(GITHUB_REPO_URL, output_dir)

skills = get_all_skills_from_dir(output_dir)
for skill in skills:
    print(skill)
    create_skill(skill)

Attempting to download repository zip via HTTP: https://github.com/kiwonilee/agentic-design-patterns-skills/archive/refs/heads/master.zip...
Repository extracted successfully from 'master' branch zip archive.
Deleted temporary zip file: /var/tmp/tmp38hm3vwp.zip
Found 28 skills.
Skill(name=parallelization, description=This skill should be used when the user wants to "run tasks in parallel", "concurrent LLM calls", "fan-out and fan-in", "parallel agents", "simultaneous execution", "reduce latency with concurrency", "parallel sub-tasks", "parallel processing", "concurrent agent execution", "run agents simultaneously", "speed up with parallelism", "async agent execution", "batch LLM calls", "parallel workflow", "parallelize agent tasks", or needs to execute multiple independent tasks simultaneously. Also responds to Korean: "병렬 작업 실행", "동시 LLM 호출", "팬아웃 팬인 패턴", "병렬 에이전트", "동시 실행", "동시성으로 지연 시간 감소", "병렬로 실행", "동시에 여러 작업", "병렬 처리", "동시에 돌려줘", "빠르게 병렬로", "여러 작업 동시 실행해줘". Apply this skill to d

In [20]:
# 스킬 목록 확인 (https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/skill-registry/create-manage#list-skills)
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

pager = client.skills.list()
for skill in pager:
    print(skill.name, skill.display_name)

projects/458778613248/locations/us-central1/skills/reflection-20260526-075146 reflection
projects/458778613248/locations/us-central1/skills/tool-use-20260526-075136 tool-use
projects/458778613248/locations/us-central1/skills/planning-20260526-075125 planning
projects/458778613248/locations/us-central1/skills/mcp-setup-20260526-075115 mcp-setup
projects/458778613248/locations/us-central1/skills/rag-20260526-075104 rag
projects/458778613248/locations/us-central1/skills/resource-aware-20260526-075053 resource-aware
projects/458778613248/locations/us-central1/skills/appendix-prompt-engineering-20260526-075043 appendix-prompt-engineering
projects/458778613248/locations/us-central1/skills/guardrails-20260526-075032 guardrails
projects/458778613248/locations/us-central1/skills/appendix-agentic-frameworks-20260526-075022 appendix-agentic-frameworks
projects/458778613248/locations/us-central1/skills/exception-handling-20260526-075011 exception-handling
projects/458778613248/locations/us-central

In [ ]:
print(f"\n--- Search Relevant Skills ---")
query = "GKE node upgrade"
print(f"Searching for skills matching query: '{query}'...")
try:
    vertexai.init(project=PROJECT_ID, location=LOCATION)
    client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

    # Call retrieve to perform semantic search
    response = client.skills.retrieve(
        query=query,
        config={"top_k": 2}
    )
    print(f"SUCCESS: Skills retrieved successfully!")
    print(f"Found {len(response.retrieved_skills)} matching skills.")
    for i, retrieved in enumerate(response.retrieved_skills):
        print(f"  [{i+1}] Skill Name: {retrieved.skill_name}")
        print(f"  [{i+1}] Description: {retrieved.description}")
except Exception as e:
    print(f"FAILED: Retrieve skills failed: {e}")